# Training a Binary Classifier for Tropical Cyclones

Now that we have the tiles and the data loader, it's time to train a classifer. To begin, we'll just classify a simple binary classifier where 0 indicates a negative tile and 1 indicates a positive tile.

## Imports

Import everything we need. 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import models
import numpy as np
from tqdm import tqdm
import random
import os
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import math
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

import sys
sys.path.append('../scripts')
from dataset import GOESDataset, create_dataloaders

## Create the Model and Trainer Classes

We need to create the model class. We'll just used pretrained ImageNet weights for a ResNet-18 model for something lightweight.

In [3]:
class CycloneClassifier(nn.Module):
    """Lightweight ResNet-18 classifier for tropical cyclone detection."""
    
    def __init__(self, num_classes=2, pretrained=True, use_three_channel=False):
        """
        Args:
            num_classes (int): Number of output classes (2 for binary: cyclone/no-cyclone)
            pretrained (bool): Whether to use pretrained ImageNet weights
            use_three_channel (bool): If False, modify conv1 for 1-channel input (no pretrained weights)
                                      If True, expect 3-channel RGB input (can use pretrained weights)
        """
        super(CycloneClassifier, self).__init__()
        
        self.use_three_channel = use_three_channel
        
        # Load ResNet-18 (lightweight option)
        if pretrained:
            self.resnet = models.resnet18(weights='DEFAULT')
        else:
            self.resnet = models.resnet18()
        
        if not use_three_channel:
            # Modify first conv layer to accept 1-channel input (grayscale satellite imagery)
            # Note: This discards pretrained weights for the first layer
            self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Otherwise, keep 3-channel input to use pretrained weights
        
        # Replace final fully connected layer
        num_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_features, num_classes)
    
    def forward(self, x):
        # If single channel, input shape is [B, 1, H, W]
        # If RGB, input shape is [B, 3, H, W]
        return self.resnet(x)


class CycloneTrainer:
    """Training pipeline for cyclone classifier."""
    
    def __init__(self, model, train_loader, val_loader, device, learning_rate=1e-3, log_file='training_log.csv', tensorboard_dir='runs'):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.log_file = log_file
        
        # TensorBoard writer
        self.writer = SummaryWriter(tensorboard_dir)
        print(f"TensorBoard logging to: {tensorboard_dir}")
        print(f"Start TensorBoard with: tensorboard --logdir={tensorboard_dir}")
        
        # Label mapping
        self.label_map = {'negative': 0, 'positive': 1}
        
        # Loss and optimizer
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=3
        )
        
        # Tracking
        self.train_losses = []
        self.val_losses = []
        self.train_accs = []
        self.val_accs = []
        self.best_val_loss = float('inf')
        self.global_step = 0
        
        # Initialize log file
        with open(self.log_file, 'w') as f:
            f.write('epoch,train_loss,train_acc,val_loss,val_acc,learning_rate\n')
    
    def train_epoch(self):
        """Train for one epoch."""
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc='Training')
        for batch_idx, batch in enumerate(pbar):
            images = batch['patch'].to(self.device)
            # Convert string labels to integers
            labels = torch.tensor([self.label_map[cat] for cat in batch['label']]).to(self.device)
            
            # Forward pass
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            self.optimizer.step()
            
            # Statistics
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Log to TensorBoard (every batch)
            self.writer.add_scalar('Loss/train_batch', loss.item(), self.global_step)
            self.writer.add_scalar('Accuracy/train_batch', 100. * predicted.eq(labels).sum().item() / labels.size(0), self.global_step)
            self.global_step += 1
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
        
        epoch_loss = running_loss / total
        epoch_acc = 100. * correct / total
        return epoch_loss, epoch_acc
    
    def validate(self):
        """Validate the model."""
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc='Validation')
            for batch in pbar:
                images = batch['patch'].to(self.device)
                # Convert string labels to integers
                labels = torch.tensor([self.label_map[cat] for cat in batch['label']]).to(self.device)
                
                # Forward pass
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                # Statistics
                running_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100.*correct/total:.2f}%'
                })
        
        epoch_loss = running_loss / total
        epoch_acc = 100. * correct / total
        return epoch_loss, epoch_acc
    
    def train(self, num_epochs, save_path='best_model.pth'):
        """Full training loop."""
        print(f"Training on device: {self.device}")
        print(f"Training samples: {len(self.train_loader.dataset)}")
        print(f"Validation samples: {len(self.val_loader.dataset)}")
        print(f"Logging to: {self.log_file}")
        
        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print("-" * 50)
            
            # Train
            train_loss, train_acc = self.train_epoch()
            self.train_losses.append(train_loss)
            self.train_accs.append(train_acc)
            
            # Validate
            val_loss, val_acc = self.validate()
            self.val_losses.append(val_loss)
            self.val_accs.append(val_acc)
            
            # Learning rate scheduling
            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Log epoch metrics to TensorBoard
            self.writer.add_scalar('Loss/train_epoch', train_loss, epoch)
            self.writer.add_scalar('Loss/val_epoch', val_loss, epoch)
            self.writer.add_scalar('Accuracy/train_epoch', train_acc, epoch)
            self.writer.add_scalar('Accuracy/val_epoch', val_acc, epoch)
            self.writer.add_scalar('Learning_Rate', current_lr, epoch)
            
            # Log to CSV file
            with open(self.log_file, 'a') as f:
                f.write(f'{epoch+1},{train_loss:.6f},{train_acc:.4f},{val_loss:.6f},{val_acc:.4f},{current_lr:.8f}\n')
            
            # Print epoch summary
            print(f"\nTrain Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
            print(f"Learning Rate: {current_lr:.8f}")
            
            # Save best model
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_loss': val_loss,
                    'val_acc': val_acc,
                }, save_path)
                print(f"✓ Saved best model (val_loss: {val_loss:.4f})")
        
        # Close TensorBoard writer
        self.writer.close()
        
        print("\n" + "="*50)
        print("Training complete!")
        print(f"Best validation loss: {self.best_val_loss:.4f}")
        
        return {
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'train_accs': self.train_accs,
            'val_accs': self.val_accs
        }


def split_dataset(dataset, val_split=0.2, seed=42):
    """
    Split dataset into train and validation sets.
    
    Args:
        dataset: PyTorch Dataset object
        val_split (float): Fraction of data to use for validation
        seed (int): Random seed for reproducibility
    
    Returns:
        train_dataset, val_dataset: Subset objects
    """
    # Set seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # Get dataset size
    dataset_size = len(dataset)
    indices = list(range(dataset_size))
    
    # Shuffle indices
    random.shuffle(indices)
    
    # Calculate split point
    split_idx = int(dataset_size * (1 - val_split))
    
    train_indices = indices[:split_idx]
    val_indices = indices[split_idx:]
    
    # Create subsets
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)
    
    print(f"Dataset split (seed={seed}):")
    print(f"  Total samples: {dataset_size}")
    print(f"  Training samples: {len(train_dataset)}")
    print(f"  Validation samples: {len(val_dataset)}")
    
    return train_dataset, val_dataset


def get_device():
    """Get the best available device (MPS for Mac, CUDA for others, CPU fallback)."""
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

In [4]:
# Set paths
metadata_path = '/Users/dylanwhite/Projects/tropical-cv/data/training/image_data.json'
weights_dir = '/Users/dylanwhite/Projects/tropical-cv/weights/resnet_classifier'
tensorboard_dir = '/Users/dylanwhite/Projects/tropical-cv/runs/'

# Set parameters
seed = 42
use_three_channel = True
label_key = 'category'
drop_classes = [-5,-4,-3,-2,-1]
batch_size = 32
num_epochs = 50
learning_rate = 1e-3
patch_size = 512
num_workers = 2
val_split = 0.2
center_bias = 0.6
evenly_sample = True

# Make output weights directory if it doesn't exist
os.makedirs(weights_dir,exist_ok=True)

# Setup device
device = get_device()
print(f"Using device: {device}")

# Create a train/test split
train_loader, val_loader = create_dataloaders(
    metadata_path, 
    batch_size=batch_size, 
    train_split=1-val_split, 
    patch_size=patch_size, 
    num_workers=num_workers, 
    center_bias=center_bias,
    label_key=label_key,
    three_channel=use_three_channel,
    drop_classes=drop_classes,
    evenly_sample=evenly_sample
)

# Create model
model = CycloneClassifier(
    num_classes=2, 
    pretrained=use_three_channel, 
    use_three_channel=use_three_channel
)

# Create trainer
if use_three_channel:
    weights_file = 'training_log_rgb.csv'
else:
    weights_file = 'training_log_grayscale.csv'
log_path = os.path.join(weights_dir,weights_file)
trainer = CycloneTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    learning_rate=learning_rate,
    log_file=log_path,
    tensorboard_dir=tensorboard_dir
)

# Train
if use_three_channel:
    weights_file = 'cyclone_classifier_rgb.pth'
else:
    weights_file = 'cyclone_classifier_grayscale.pth'
weights_path = os.path.join(weights_dir,weights_file)
history = trainer.train(num_epochs=num_epochs, save_path=weights_path)

print(f"\nTraining log saved to '{log_path}'")
print(f"Best model saved to '{weights_path}'")

Using device: mps
Train samples: 2294, Test samples: 574
TensorBoard logging to: /Users/dylanwhite/Projects/tropical-cv/runs/
Start TensorBoard with: tensorboard --logdir=/Users/dylanwhite/Projects/tropical-cv/runs/
Training on device: mps
Training samples: 2294
Validation samples: 574
Logging to: /Users/dylanwhite/Projects/tropical-cv/weights/resnet_classifier/training_log_rgb.csv

Epoch 1/50
--------------------------------------------------


Validation:  17%|█▋        | 3/18 [00:09<00:35,  2.35s/it, loss=0.7201, acc=35.42%]/Users/dylanwhite/Projects/tropical-cv/notebooks/../scripts/dataset.py:333: RuntimeWarning: invalid value encountered in divide
  rad_normalized = (rad_data - np.nanmin(rad_data)) / (np.nanmax(rad_data) - np.nanmin(rad_data))
/Users/dylanwhite/Projects/tropical-cv/notebooks/../scripts/dataset.py:335: RuntimeWarning: invalid value encountered in cast
  rad_uint8 = (rad_normalized * 255).astype(np.uint8)
Validation: 100%|██████████| 18/18 [00:33<00:00,  1.84s/it, loss=0.7166, acc=36.41%]



Train Loss: 0.6798 | Train Acc: 56.15%
Val Loss: 0.7170 | Val Acc: 36.41%
Learning Rate: 0.00100000
✓ Saved best model (val_loss: 0.7170)

Epoch 2/50
--------------------------------------------------


Training:  61%|██████    | 44/72 [02:49<01:44,  3.74s/it, loss=0.6787, acc=50.28%]libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x10d84ef20>
Traceback (most recent call last):
  File "/Users/dylanwhite/Projects/tropical-cv/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1653, in __del__
    def __del__(self):

  File "/Users/dylanwhite/Projects/tropical-cv/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/signal_handling.py", line 73, in handler
    _error_if_any_worker_fails()
RuntimeError: DataLoader worker (pid 32523) is killed by signal: Abort trap: 6. 
Training:  61%|██████    | 44/72 [02:53<01:50,  3.94s/it, loss=0.6787, acc=50.28%]


KeyboardInterrupt: 

In [ ]:
def get_model_predictions(model, dataloader, device, label_map={'negative': 0, 'positive': 1}):
    """
    Get all predictions and confidences from the model.
    
    Returns:
        predictions: list of dicts with 'patch', 'true_label', 'pred_label', 'confidence', 'pred_probs'
    """
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader):
            images = batch['patch'].to(device)
            true_labels = torch.tensor([label_map[cat] for cat in batch['label']])
            
            # Get model predictions
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)  # Convert logits to probabilities
            confidences, pred_labels = probs.max(1)
            
            # Store results
            for i in range(images.size(0)):
                all_predictions.append({
                    'patch': batch['patch'][i].cpu(),  # Keep on CPU
                    'true_label': true_labels[i].item(),
                    'pred_label': pred_labels[i].cpu().item(),
                    'confidence': confidences[i].cpu().item(),
                    'pred_probs': probs[i].cpu().numpy(),
                    'category_str': batch['label'][i]
                })
    
    return all_predictions


def filter_predictions(predictions, prediction_type):
    """
    Filter predictions by type: 'TP', 'TN', 'FP', 'FN'
    
    Args:
        predictions: list from get_model_predictions()
        prediction_type: 'TP', 'TN', 'FP', or 'FN'
    
    Returns:
        filtered and sorted by confidence (descending)
    """
    filtered = []
    
    for pred in predictions:
        true_label = pred['true_label']
        pred_label = pred['pred_label']
        
        if prediction_type == 'TP':  # True Positive: predicted positive, actually positive
            if pred_label == 1 and true_label == 1:
                filtered.append(pred)
        elif prediction_type == 'TN':  # True Negative: predicted negative, actually negative
            if pred_label == 0 and true_label == 0:
                filtered.append(pred)
        elif prediction_type == 'FP':  # False Positive: predicted positive, actually negative
            if pred_label == 1 and true_label == 0:
                filtered.append(pred)
        elif prediction_type == 'FN':  # False Negative: predicted negative, actually positive
            if pred_label == 0 and true_label == 1:
                filtered.append(pred)
    
    # Sort by confidence (descending)
    filtered.sort(key=lambda x: x['confidence'], reverse=True)
    
    return filtered


def visualize_predictions(predictions, num_images=16, title="Predictions"):
    """
    Create a mosaic visualization of predictions.
    
    Args:
        predictions: list of prediction dicts (from filter_predictions)
        num_images: number of images to show
        title: title for the plot
    """
    if len(predictions) == 0:
        print(f"No predictions found for {title}")
        return
    
    # Limit to available predictions
    num_images = min(num_images, len(predictions))
    
    # Get patch size from first image
    patch_size = predictions[0]['patch'].shape[-1]
    
    # Calculate mosaic dimensions
    padding = 3
    mosaic_width = 4
    mosaic_height = math.ceil(num_images / mosaic_width)
    mosaic_img_width = mosaic_width * patch_size + (mosaic_width + 1) * padding
    mosaic_img_height = mosaic_height * patch_size + (mosaic_height + 1) * padding
    
    # Create empty mosaic array
    # Handle both grayscale and RGB
    if predictions[0]['patch'].shape[0] == 3:  # RGB
        mosaic = np.zeros((mosaic_img_height, mosaic_img_width, 3), dtype=np.uint8)
    else:  # Grayscale
        mosaic = np.zeros((mosaic_img_height, mosaic_img_width), dtype=np.uint8)
    
    # Fill mosaic
    for idx in range(num_images):
        i = idx % mosaic_width
        j = idx // mosaic_width
        
        pred = predictions[idx]
        patch = pred['patch']
        
        # Convert to numpy and scale to 0-255
        if patch.dim() == 3:
            if patch.shape[0] == 3:  # RGB [3, H, W]
                patch_np = patch.permute(1, 2, 0).numpy()  # [H, W, 3]
            else:  # Grayscale [1, H, W]
                patch_np = patch.squeeze(0).numpy()  # [H, W]
        else:
            patch_np = patch.numpy()
        
        # Scale to 0-255
        patch_np = (255 - patch_np * 255).astype(np.uint8)
        
        # Calculate position in mosaic
        y_start = padding + j * (patch_size + padding)
        x_start = padding + i * (patch_size + padding)
        
        # Place tile in mosaic
        mosaic[y_start:y_start + patch_size, x_start:x_start + patch_size] = patch_np
    
    # Convert to PIL Image
    if len(mosaic.shape) == 2:  # Grayscale
        mosaic_img = Image.fromarray(mosaic)
    else:  # RGB
        mosaic_img = Image.fromarray(mosaic)
    
    # Add text labels
    draw = ImageDraw.Draw(mosaic_img)
    
    # Try to load a font
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 24)
    except:
        font = ImageFont.load_default(size=24)
    
    # Add text to each patch
    for idx in range(num_images):
        i = idx % mosaic_width
        j = idx // mosaic_width
        
        pred = predictions[idx]
        true_label = pred['true_label']
        pred_label = pred['pred_label']
        confidence = pred['confidence']
        
        # Create label text
        label_names = {0: 'negative', 1: 'positive'}
        text = f"True: {label_names[true_label]}\nPred: {label_names[pred_label]}\nConf: {confidence:.2f}"
        
        # Calculate position for text
        x_start = padding + i * (patch_size + padding) + 5
        y_start = padding + j * (patch_size + padding) + 5
        
        # Draw text with outline
        for line_num, line in enumerate(text.split('\n')):
            y_pos = y_start + line_num * 28
            # Black outline
            for offset_x in [-1, 0, 1]:
                for offset_y in [-1, 0, 1]:
                    draw.text((x_start + offset_x, y_pos + offset_y), line, fill=0, font=font)
            # White text
            draw.text((x_start, y_pos), line, fill=255, font=font)
    
    # Display
    plt.figure(figsize=(15, 15 * mosaic_height / mosaic_width))
    if len(mosaic.shape) == 2:
        plt.imshow(mosaic_img, cmap="Greys")
    else:
        plt.imshow(mosaic_img)
    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Load your model
model = CycloneClassifier(num_classes=2, pretrained=True, use_three_channel=True)
checkpoint = torch.load(weights_path)
model.load_state_dict(checkpoint['model_state_dict'])

device = get_device()
model = model.to(device)

# Create test/validation dataloader
# (use your validation dataset or a separate test set)

# Get all predictions
print("Getting model predictions...")
all_predictions = get_model_predictions(model, val_loader, device)

In [ ]:
# Visualize different types
print("\nMost Confident True Positives:")
tp_predictions = filter_predictions(all_predictions, 'TP')
print(f"Found {len(tp_predictions)} True Positives")
visualize_predictions(tp_predictions, num_images=16, title="Most Confident True Positives (Correct)")

In [ ]:
print("\nMost Confident False Positives:")
fp_predictions = filter_predictions(all_predictions, 'FP')
print(f"Found {len(fp_predictions)} False Positives")
visualize_predictions(fp_predictions, num_images=16, title="Most Confident False Positives (Wrong!)")

In [ ]:
print("Most Confident False Negatives:")
fn_predictions = filter_predictions(all_predictions, 'FN')
print(f"Found {len(fn_predictions)} False Negatives")
visualize_predictions(fn_predictions, num_images=16, title="Most Confident False Negatives (Missed Cyclones)")

In [ ]:
print("Most Confident True Negatives:")
tn_predictions = filter_predictions(all_predictions, 'TN')
print(f"Found {len(tn_predictions)} True Negatives")
visualize_predictions(tn_predictions, num_images=16, title="Most Confident True Negatives (Correct)")

In [ ]:
def plot_confusion_matrix(predictions, normalize=False, figsize=(8, 6)):
    """
    Plot confusion matrix from model predictions.
    
    Args:
        predictions: list from get_model_predictions()
        normalize: if True, show percentages instead of counts
        figsize: figure size
    """
    # Extract true and predicted labels
    y_true = [pred['true_label'] for pred in predictions]
    y_pred = [pred['pred_label'] for pred in predictions]
    
    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Normalize if requested
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2%'
        title = 'Normalized Confusion Matrix'
    else:
        fmt = 'd'
        title = 'Confusion Matrix'
    
    # Create figure
    plt.figure(figsize=figsize)
    
    # Plot heatmap
    sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues', 
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                cbar_kws={'label': 'Percentage' if normalize else 'Count'})
    
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title(title)
    
    # Add counts in each cell if normalized
    if normalize:
        # Re-compute raw counts
        cm_counts = confusion_matrix(y_true, y_pred)
        # Add count annotations
        for i in range(2):
            for j in range(2):
                count = cm_counts[i, j]
                plt.text(j + 0.5, i + 0.7, f'(n={count})', 
                        ha='center', va='center', fontsize=10, color='gray')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed metrics
    print("\n" + "="*50)
    print("CLASSIFICATION REPORT")
    print("="*50)
    print(classification_report(y_true, y_pred, 
                                target_names=['Negative', 'Positive'],
                                digits=4))
    
    # Print confusion matrix breakdown
    tn, fp, fn, tp = cm.ravel() if not normalize else confusion_matrix(y_true, y_pred).ravel()
    
    print("\n" + "="*50)
    print("CONFUSION MATRIX BREAKDOWN")
    print("="*50)
    print(f"True Negatives (TN):  {tn:5d}  (Correctly predicted no cyclone)")
    print(f"False Positives (FP): {fp:5d}  (Incorrectly predicted cyclone)")
    print(f"False Negatives (FN): {fn:5d}  (Missed cyclone)")
    print(f"True Positives (TP):  {tp:5d}  (Correctly predicted cyclone)")
    print("-"*50)
    print(f"Total samples:        {tn+fp+fn+tp:5d}")
    
    # Calculate additional metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print("\n" + "="*50)
    print("KEY METRICS")
    print("="*50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}  (Of predicted cyclones, how many were correct?)")
    print(f"Recall:    {recall:.4f}  (Of actual cyclones, how many did we find?)")
    print(f"F1-Score:  {f1:.4f}  (Harmonic mean of precision and recall)")
    
    return cm


def plot_confidence_distribution(predictions):
    """
    Plot distribution of model confidence for correct vs incorrect predictions.
    """
    # Separate correct and incorrect predictions
    correct_conf = [p['confidence'] for p in predictions if p['true_label'] == p['pred_label']]
    incorrect_conf = [p['confidence'] for p in predictions if p['true_label'] != p['pred_label']]
    
    plt.figure(figsize=(10, 6))
    
    # Plot histograms
    plt.hist(correct_conf, bins=20, alpha=0.6, label='Correct Predictions', color='green', edgecolor='black')
    plt.hist(incorrect_conf, bins=20, alpha=0.6, label='Incorrect Predictions', color='red', edgecolor='black')
    
    plt.xlabel('Confidence', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.title('Model Confidence Distribution', fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*50)
    print("CONFIDENCE STATISTICS")
    print("="*50)
    print(f"Correct predictions - Mean confidence:   {np.mean(correct_conf):.4f}")
    print(f"Incorrect predictions - Mean confidence: {np.mean(incorrect_conf):.4f}")
    print(f"\nTotal correct:   {len(correct_conf)}")
    print(f"Total incorrect: {len(incorrect_conf)}")

In [ ]:
# Plot confusion matrix (raw counts)
print("\n1. Confusion Matrix (Counts)")
cm = plot_confusion_matrix(all_predictions, normalize=False)

# Plot normalized confusion matrix (percentages)
print("\n2. Confusion Matrix (Normalized)")
cm_norm = plot_confusion_matrix(all_predictions, normalize=True)

# Plot confidence distribution
print("\n3. Confidence Distribution")
plot_confidence_distribution(all_predictions)